# Designing novel small proteins with Evo 2, in context

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evo-design/evo2_flash/blob/main/notebooks/small_protein_ICL/small_protein_ICL.ipynb)

Show Evo 2 a handful of members of a protein family, separated by a fixed
nucleotide spacer, and let it continue the sequence. The continuation is a new coding
sequence. **No fine-tuning, no structure model, no optimizer** — the design signal is
entirely in the prompt.

This notebook runs that experiment end to end on two families and asks one question:

> **How many generated proteins are both _coherent_ (a real member of the family) and
> _novel_ (not a copy of anything in the pool), and how does that depend on the number of
> in-context examples?**

| family | pool | lengths | what it is |
|---|---|---|---|
| **beta-lactamase** (`bla`) | 64 | 242–297 aa | UniRef90 homologs, length-filtered |
| **Top7** (`top7`) | 196 | 92 aa (all) | ProteinMPNN redesigns of the de novo fold Top7 |


**Output:** `novel_yield_T0.8.png` — novel + coherent yield against in-context example
count, one line per family.

---

### Hardware and time

- **GPU:** one with ≥ 24 GB — Colab **A100** or **L4**. A T4 is both too small and too
  old. Evo 2 7B is ~13 GB of weights, and the longest prompt here is 54 kb of DNA.
- **Disk:** ~15 GB for the checkpoint.
- **Time:** ~10 min to download and load the model, then ~15–25 min of generation and
  search at the default scale on an A100. Everything after generation is CPU-only and
  takes under a minute, so the figure is cheap to re-cut.

`N_PROMPTS` in the config cell trades error bars for wall-clock, and dropping the largest
`n` from either family's grid roughly halves that family's cost.

## 1 · Environment

First confirm the runtime actually has a GPU, then install. **Runtime → Change runtime
type → A100 GPU** (or L4) if the next cell reports no device.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)

### Install

**Two environments, on purpose.** Colab's kernel is Python 3.13 and its version cannot be
changed. evo2-flash requires `<3.13`, and vLLM 0.26 needs a specific CUDA 12.9 torch build
that would collide with Colab's preinstalled one. So the engine gets a dedicated Python
3.12 venv (`uv venv`) and the kernel is left alone — nothing to restart, nothing
downgraded. Cells that need the engine call that venv's Python; the analysis half runs in
the kernel. ~6 minutes, once per runtime.

In [ ]:
ENVDIR = "/content/evo2env"

# vLLM 0.26.0, CUDA 12.9 build: runs on CUDA 12 and CUDA 13 drivers. PyPI's default vllm
# build is CUDA 13 only and cannot use the GPU on a CUDA 12 driver.
VLLM_WHEEL = ("https://github.com/vllm-project/vllm/releases/download/v0.26.0/"
              "vllm-0.26.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl")

!pip install -q uv
!uv venv -q --seed --python 3.12 {ENVDIR}

# The torch stack pinned to the exact +cu129 builds of the A100/L4 reference runs. vLLM
# declares a plain `torch==2.11.0`, which PyPI serves as a CUDA 13 build -- that pulls
# nvidia-cuda-runtime (cu13) rather than nvidia-cuda-runtime-cu12, and vLLM's cu129
# extension then dies with "libcudart.so.12: cannot open shared object file". A +cu129
# local version can only come from the cu129 index, so these pins are unambiguous.
!{ENVDIR}/bin/pip install -q torch==2.11.0+cu129 torchvision==0.26.0+cu129 \
    torchaudio==2.11.0+cu129 torchcodec==0.15.0+cu129 \
    --extra-index-url https://download.pytorch.org/whl/cu129

!{ENVDIR}/bin/pip install -q "{VLLM_WHEEL}" \
    --extra-index-url https://download.pytorch.org/whl/cu129

# ninja: FlashInfer JIT-builds its kernels with it. The venv is 3.12, so evo2-flash
# installs without --ignore-requires-python.
!{ENVDIR}/bin/pip install -q ninja evo2-flash

# The analysis half runs in the Colab kernel and needs only Biopython on top of what is
# already there (numpy, pandas, matplotlib).
!pip install -q biopython

# MMseqs2, static build -> ./mmseqs/bin/mmseqs
!wget -qO- https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz | tar xz

In [ ]:
import os, shutil, subprocess

PY = f"{ENVDIR}/bin/python"
os.environ.update(
    VLLM_ENABLE_V1_MULTIPROCESSING="0",   # the Evo2 API needs the engine in-process
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
    PATH=f"{ENVDIR}/bin:/usr/local/cuda/bin:" + os.environ["PATH"],
)

MMSEQS = shutil.which("mmseqs") or os.path.abspath("mmseqs/bin/mmseqs")
assert os.path.exists(MMSEQS), "mmseqs not found -- re-run the install cell"
print(subprocess.run([MMSEQS, "version"], capture_output=True, text=True).stdout.strip())

CHECK = """
import sys, torch, vllm, evo2_flash
assert "+cu12" in torch.__version__, (
    f"torch is {torch.__version__}; vLLM's cu129 build needs a +cu129 torch, or it fails "
    f"with 'libcudart.so.12: cannot open shared object file'. Re-run the install cell.")
print(f"venv: python {sys.version.split()[0]} | torch {torch.__version__} "
      f"| vllm {vllm.__version__} | cuda {torch.cuda.is_available()}")
"""
r = subprocess.run([PY, "-c", CHECK], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-3000:])
r.check_returncode()

> **No runtime restart needed.** Colab's own packages were not touched — everything the
> engine needs lives in `/content/evo2env`.

## 2 · The two pools

Each family ships as a protein FASTA plus a **codon cache**: the same pool already
codon-optimized for *E. coli* by
[CodonTransformer](https://github.com/Adibvafa/CodonTransformer). Evo 2 is a nucleotide
model, so the prompt has to be DNA.

In [ ]:
import json, pathlib, urllib.request

RAW = ("https://raw.githubusercontent.com/evo-design/evo2_flash/main/"
       "notebooks/small_protein_ICL/data")
DATA = pathlib.Path("data")
DATA.mkdir(exist_ok=True)

for name in ["beta_lactamase_filtered.fasta", "bla_codons.json",
             "top7_mpnn.fasta", "top7_codons.json"]:
    if not (DATA / name).exists():                 # already present when run from a clone
        urllib.request.urlretrieve(f"{RAW}/{name}", DATA / name)
        print(f"downloaded {name}")


def load_pool(path):
    """Protein sequences in FASTA order, cleaned as the reference pipeline does.

    bla is wrapped at 60 columns, so a line-based loader would return 320 fragments
    instead of 64 proteins. Order matters: the example draw indexes into this list.
    """
    seqs, cur = [], None
    for line in open(path):
        if line.startswith(">"):
            if cur:
                seqs.append(cur)
            cur = ""
        elif cur is not None:
            cur += line.strip()
    if cur:
        seqs.append(cur)
    return [s for s in (x.upper().replace("*", "") for x in seqs) if s]

## 3 · Configuration


The prompt is
`SPACER s1 SPACER s2 … SPACER sK SPACER`


In [ ]:
#@title Experiment configuration

TEMPERATURE        = 0.8     # decoding temperature
TOP_K              = 4       # the reference pipeline's value; bounds what T can do
TOP_P              = 1.0     # off (vortex's top-p filter short-circuits at 1.0)

SPACER             = "GGGGGGGG"
SEED               = 1337    # example-selection seed; draw p uses SEED + p
N_PROMPTS          = 10      # independent example draws per cell (published sweep: 25)
SAMPLES_PER_PROMPT = 4       # -> 40 samples per cell

MAX_ID             = 85.0    # novel = max %ID to the POOL below this
GPU_MEM_FRACTION   = 0.85

FAMILIES = {
    "bla": dict(
        label="beta-lactamase",
        fasta="beta_lactamase_filtered.fasta",
        codons="bla_codons.json",
        n_examples=[2, 4, 8, 16, 32, 64],
        evalue=0.01,         # MMseqs significance for "is a family member"
    ),
    "top7": dict(
        label="Top7 redesigns",
        fasta="top7_mpnn.fasta",
        codons="top7_codons.json",
        n_examples=[2, 4, 8, 16, 32, 64, 128],
        evalue=1.0,          # a 92-aa de novo design needs a permissive threshold
    ),
}

## 4 · Build the prompts

In [ ]:
import random


def select_examples(pool, n, seed):
    random.seed(seed)
    return random.sample(pool, min(n, len(pool)))


def compose_prompt(dna_seqs, spacer=SPACER):
    """left layout: a spacer before every example, plus a trailing one."""
    return "".join(spacer + s for s in dna_seqs) + spacer


def extract_cds(continuation, spacer=SPACER):
    """The prompt ended on a separator, so the CDS is everything before the next one."""
    return continuation.split(spacer)[0] if spacer in continuation else None


def build_cells(fam_key, cache):
    proteins, dna = cache["proteins"], cache["dna"]
    assert len(set(proteins)) == len(proteins), f"{fam_key}: duplicate pool sequences"
    dna_of = dict(zip(proteins, dna))

    cells = []
    for n in FAMILIES[fam_key]["n_examples"]:
        if n > len(proteins):
            print(f"  {fam_key}: skipping n={n} (pool has {len(proteins)})")
            continue
        draws = []
        for p in range(N_PROMPTS):
            sel = select_examples(proteins, n, SEED + p)
            prompt = compose_prompt([dna_of[a] for a in sel])
            assert prompt.count(SPACER) == n + 1
            assert prompt.startswith(SPACER) and prompt.endswith(SPACER)
            draws.append(dict(prompt_idx=p, prompt=prompt,
                              gen_length=max(map(len, sel)) * 3 + 30))
        cells.append(dict(
            family=fam_key, n_examples=n, draws=draws,
            pool_lo=min(map(len, proteins)), pool_hi=max(map(len, proteins)),
            evalue=FAMILIES[fam_key]["evalue"],
            # One budget per cell, so the whole cell goes into one batched call. Under the
            # left layout a bigger budget can only let a late first separator still be
            # found; it never changes an already-extracted CDS.
            gen_length=max(d["gen_length"] for d in draws),
        ))
    return cells


CELLS = []
for fam, info in FAMILIES.items():
    cache = json.loads((DATA / info["codons"]).read_text())
    assert cache["proteins"] == load_pool(DATA / info["fasta"]), \
        f"{fam}: codon cache does not match {info['fasta']}"
    lens = [len(s) for s in cache["proteins"]]
    print(f"{fam:>5}  {len(lens)} proteins, {min(lens)}-{max(lens)} aa, "
          f"codon-optimized for {cache['organism']}")
    CELLS += build_cells(fam, cache)

prefill = sum(len(d["prompt"]) for c in CELLS for d in c["draws"])
decode = sum(c["gen_length"] * N_PROMPTS * SAMPLES_PER_PROMPT for c in CELLS)
longest = max(len(d["prompt"]) + c["gen_length"] for c in CELLS for d in c["draws"])
print(f"\n{len(CELLS)} cells x {N_PROMPTS} draws x {SAMPLES_PER_PROMPT} samples = "
      f"{len(CELLS) * N_PROMPTS * SAMPLES_PER_PROMPT} sequences")
print(f"  unique prefill {prefill:,} nt   decode {decode:,} nt   "
      f"longest context {longest:,} nt")

## 5 · Generate

The engine lives in the venv, so this stage runs as a script rather than in the kernel:
the prompts go to `work/cells.json`, the venv's Python generates, and the continuations
come back as JSON. That is the same file hand-off the original four-stage pipeline uses,
for the same reason — the generation and analysis environments cannot share a process.

The first run downloads ~15 GB for the checkpoint. Allow ~60 GB of free disk: the
download and the conversion both need room, and conversion peaks around 16 GB of RAM, so
use a **High-RAM** runtime.

In [ ]:
GEN_SCRIPT = r"""
# Generation stage -- runs in the evo2env venv, not the Colab kernel.
import json, pathlib, time
from evo2_flash import Evo2

WORK = pathlib.Path("work")
cfg = json.loads((WORK / "cells.json").read_text())
cells = cfg["cells"]
longest = max(len(p) + c["gen_length"] for c in cells for p in c["prompts"])

model = Evo2(cfg["model"], gpu_memory_utilization=cfg["gpu_memory_utilization"])
print(f"max_model_len = {model.max_model_len:,}", flush=True)
if model.max_model_len < longest:
    raise SystemExit(
        f"the longest cell needs {longest:,} positions but max_model_len is "
        f"{model.max_model_len:,}. Drop the largest n from a family's grid, or pass "
        f"Evo2(..., kv_cache_dtype='fp8') to double KV capacity.")

t0 = time.time()
for i, c in enumerate(cells, 1):
    t = time.time()
    out = model.generate(
        c["prompts"], n_tokens=c["gen_length"],
        temperature=cfg["temperature"], top_k=cfg["top_k"], top_p=cfg["top_p"],
        seed=cfg["seed"] + 1000 * i,   # a distinct sampling stream per cell
        verbose=0,                     # verbose >= 1 echoes every 54 kb prompt
    )
    (WORK / f"gen_{c['family']}_{c['n_examples']:03d}.json").write_text(
        json.dumps({"sequences": out.sequences}))
    tok = sum(len(p) for p in c["prompts"])
    hit = 100 * sum(out.num_cached_tokens) / max(tok, 1)
    ideal = 100 * (1 - c["uniq_tok"] / max(tok, 1))
    print(f"[{i:>2}/{len(cells)}] {c['family']:>5} n={c['n_examples']:<4} "
          f"{len(c['prompts'])} seqs x {c['gen_length']} nt in {time.time() - t:6.1f}s   "
          f"prefix cache {hit:4.1f}% (ideal {ideal:4.1f}%)", flush=True)
print(f"\ntotal {time.time() - t0:.0f}s", flush=True)
"""

WORK = pathlib.Path("work")
WORK.mkdir(exist_ok=True)
(WORK / "cells.json").write_text(json.dumps(dict(
    model="evo2_7b", gpu_memory_utilization=GPU_MEM_FRACTION,
    temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P, seed=SEED,
    cells=[dict(
        family=c["family"], n_examples=c["n_examples"], gen_length=c["gen_length"],
        uniq_tok=sum(len(d["prompt"]) for d in c["draws"]),
        prompts=[d["prompt"] for d in c["draws"] for _ in range(SAMPLES_PER_PROMPT)],
    ) for c in CELLS],
)))
(WORK / "generate.py").write_text(GEN_SCRIPT)

subprocess.run([PY, str(WORK / "generate.py")], check=True)

# Back into the kernel, in exactly the shape the analysis cells expect.
for c in CELLS:
    g = json.loads((WORK / f"gen_{c['family']}_{c['n_examples']:03d}.json").read_text())
    c["sequences"] = g["sequences"]
    c["prompt_of_seq"] = [d["prompt_idx"] for d in c["draws"]
                          for _ in range(SAMPLES_PER_PROMPT)]
print(f"\nloaded {sum(len(c['sequences']) for c in CELLS)} sequences into the kernel")

## 6 · Extract, translate, search

Three filters, in order. A design is **coherent** when it clears all three:

1. **parses** — the continuation emitted a closing separator, so a CDS can be cut out of
   it. This is the window-free read on whether the model learned the record format;
2. **lands in the pool length window** and is free of premature stops — an ORF that
   terminates early is not a protein of the length the window is checking, and it inflates
   apparent novelty by truncating the real product;
3. **hits the pool** under MMseqs2 below the family e-value — the argument that the design
   is a family member rather than plausible-looking noise.

In [ ]:
import csv, subprocess, tempfile, warnings
from Bio import BiopythonWarning
from Bio.Seq import Seq

warnings.simplefilter("ignore", BiopythonWarning)      # partial final codon
FIELDS = ("query,target,pident,alnlen,mismatch,gapopen,"
          "qstart,qend,tstart,tend,evalue,bits")
WORK = pathlib.Path("work")
WORK.mkdir(exist_ok=True)

for c in CELLS:
    designs = []
    for k, (nt, pidx) in enumerate(zip(c["sequences"], c["prompt_of_seq"]), 1):
        cds = extract_cds(nt.upper())
        if cds is None:
            continue
        aa = str(Seq(cds).translate())
        designs.append(dict(qid=f"{c['family']}_{c['n_examples']}_seq_{k}",
                            prompt_idx=pidx, aa=aa,
                            mature_len=len(aa.rstrip("*")),
                            internal_stop="*" in aa.rstrip("*")))
    c["designs"] = designs

    stem = WORK / f"{c['family']}_{c['n_examples']:03d}"
    query = stem.with_suffix(".faa")
    query.write_text("".join(f">{d['qid']}\n{d['aa']}\n" for d in designs))
    c["m8"] = stem.with_suffix(".m8")
    if designs:
        with tempfile.TemporaryDirectory() as tmp:
            r = subprocess.run(
                [MMSEQS, "easy-search", str(query),
                 str(DATA / FAMILIES[c["family"]]["fasta"]), str(c["m8"]), tmp,
                 "-e", "10", "--format-output", FIELDS],
                capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError(r.stdout[-2000:] + r.stderr[-2000:])
    else:
        c["m8"].write_text("")
    print(f"{c['family']:>5} n={c['n_examples']:<4} {len(c['sequences']):>3} samples -> "
          f"{len(designs):>3} parsed -> {sum(1 for _ in open(c['m8'])):>6} "
          f"alignment rows")

## 7 · Score

`novel` = coherent **and** max %ID to the pool below `MAX_ID`. Scored per draw and then
averaged, so each cell gets a mean and an SEM instead of a single number.

In [ ]:
import numpy as np
import pandas as pd

rows, funnel = [], []
for c in CELLS:
    significant, best_id = set(), {}
    for r in csv.reader(open(c["m8"]), delimiter="\t"):
        if not r:
            continue
        best_id[r[0]] = max(best_id.get(r[0], 0.0), float(r[2]))    # %ID to the pool
        if float(r[10]) < c["evalue"]:
            significant.add(r[0])

    per_draw = np.zeros((N_PROMPTS, 2), dtype=int)                  # coherent, novel
    n_window = 0
    for d in c["designs"]:
        in_window = (c["pool_lo"] <= d["mature_len"] <= c["pool_hi"]
                     and not d["internal_stop"])
        n_window += in_window
        if not (in_window and d["qid"] in significant):
            continue
        per_draw[d["prompt_idx"], 0] += 1
        per_draw[d["prompt_idx"], 1] += best_id.get(d["qid"], 0.0) < MAX_ID

    y = 100 * per_draw[:, 0] / SAMPLES_PER_PROMPT
    ny = 100 * per_draw[:, 1] / SAMPLES_PER_PROMPT
    sem = ny.std(ddof=1) / np.sqrt(len(ny)) if len(ny) > 1 else 0.0
    rows.append(dict(
        family=c["family"], n_examples=c["n_examples"], n_prompts=N_PROMPTS,
        yield_mean=round(y.mean(), 1),
        novel_yield_mean=round(ny.mean(), 1),
        novel_yield_sem=round(sem, 1),
        kept_pct=round(100 * ny.sum() / max(y.sum(), 1e-9), 1)))
    funnel.append(dict(
        family=c["family"], n=c["n_examples"], samples=len(c["sequences"]),
        parsed=len(c["designs"]), in_window=n_window,
        coherent=int(per_draw[:, 0].sum()), novel=int(per_draw[:, 1].sum())))

df = pd.DataFrame(rows)
df.to_csv("novel_yield_T0.8.tsv", sep="\t", index=False)

print("funnel  (samples -> parsed -> length window -> family hit -> novel)\n")
print(pd.DataFrame(funnel).to_string(index=False))
print("\n\nnovel + coherent yield, % of samples (mean, SEM over draws)\n")
print(df.to_string(index=False))

## 8 · The figure

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#dcdcd8"
SERIES = {"bla": "#2a78d6", "top7": "#eb6834"}       # categorical slots 1 and 2
GRIDN = sorted({n for f in FAMILIES.values() for n in f["n_examples"]})

fig, ax = plt.subplots(figsize=(7.8, 4.9))
# Explicit margins, so the header block below sits in reserved space rather than
# wherever tight_layout happens to leave room.
fig.subplots_adjust(left=0.115, right=0.805, top=0.775, bottom=0.135)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
ax.grid(True, color=GRID, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color(GRID)
    ax.spines[s].set_linewidth(1)
ax.tick_params(colors=INK_2, labelsize=10, length=4, width=1)

for fam, info in FAMILIES.items():
    g = df[df.family == fam].sort_values("n_examples")
    if g.empty:
        continue
    color = SERIES[fam]
    ax.errorbar(g.n_examples, g.novel_yield_mean, yerr=g.novel_yield_sem,
                color=color, linewidth=2, solid_joinstyle="round",
                solid_capstyle="round", marker="o", markersize=8,
                markerfacecolor=color, markeredgecolor=SURFACE, markeredgewidth=2,
                elinewidth=1.5, capsize=0, ecolor=color, label=info["label"], zorder=3)
    # Direct label at the line end, in text ink: identity is never colour-alone.
    last = g.iloc[-1]
    ax.annotate(info["label"], xy=(last.n_examples, last.novel_yield_mean),
                xytext=(10, 0), textcoords="offset points", fontsize=9.5,
                color=INK_2, ha="left", va="center")

ax.set_xscale("log", base=2)
ax.set_xticks(GRIDN)
ax.set_xticklabels([str(n) for n in GRIDN], fontsize=9)
ax.minorticks_off()
ax.set_xlim(1.8, 190)
ax.set_ylim(-3, 100)
ax.set_yticks(range(0, 101, 20))
ax.set_yticklabels([f"{v}%" for v in range(0, 101, 20)])
ax.set_xlabel("In-context examples", fontsize=10, color=INK_2, labelpad=6)
ax.set_ylabel("Novel + coherent yield\n(% of samples)", fontsize=10.5, color=INK_2,
              labelpad=8)

leg = ax.legend(loc="upper left", frameon=False, fontsize=9.5, handlelength=1.6,
                borderpad=0, labelspacing=0.4)
for tx in leg.get_texts():
    tx.set_color(INK_2)

for ext in ("png", "svg"):
    fig.savefig(f"novel_yield_T0.8.{ext}", facecolor=SURFACE, dpi=150,
                bbox_inches="tight")
plt.show()